Session 1


In [2]:
# Standard ML Classifier
def predict_spam(email_text):
    # Imagine a trained model here
    return {"label": "SPAM", "confidence": 0.98}

result = predict_spam("You won $1,000,000!")
print(result)
# Output: {'label': 'SPAM', 'confidence': 0.98}
# End of program. Stops here. Lame.

{'label': 'SPAM', 'confidence': 0.98}


In [3]:
# Agentic Workflow
# Moving from print(result) to perform_action(result)

def agentic_handler(email_text):
    if predict_spam(email_text)['confidence'] > 0.9:
        # THE DIFFERENCE: It takes ACTION
        action = llm.ask("Draft a polite 'I'm busy' reply and move to trash.")
        return action


Session 2: Python Hands-On - Your First AI Function Call

Step1

In [20]:
import os
from google.colab import userdata # Import userdata for Colab secrets
from groq import Groq

# Retrieve API key from Colab secrets, assuming it's named 'GROQ_API_KEY'
api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

def ask_llm(question):
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": question}]
        )
        return response.choices[0].message.content
    except Exception as e:
        # This prints the FULL error details
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Message: {str(e)}")

        # If it's an API error, get more details
        if hasattr(e, 'response'):
            print(f"Status Code: {e.response.status_code}")
            print(f"Response Body: {e.response.text}")
        return None

# Test
result = ask_llm("Whats the capital of France ?'")
if result:
    print(f"SUCCESS: {result}")

SUCCESS: The capital of France is Paris.


In [9]:
pip install groq

Step 2

In [15]:
# Test
result = ask_llm("Whats the Weather in Bangalore ?'")
if result:
    print(f"SUCCESS: {result}")

SUCCESS: I'm not capable of real-time weather updates. However, based on general climate patterns in Bangalore, I can give you some information.

Bangalore is known for its tropical savanna climate, which is characterized by:

- Hot summers from March to May with temperatures often reaching 35-36°C (95-97°F)
- Mild winters from December to February with temperatures usually ranging from 15-20°C (59-68°F)
- Moderate rainy season from June to September with most of the annual rainfall occurring during this period

To get the current weather in Bangalore, I recommend checking an online weather forecasting service or website, such as AccuWeather, Weather.com, or the India Meteorological Department (IMD) website.

If you need specific information about the weather in Bangalore for a particular day or period, please provide the date or the time period for which you need the information, and I will try to assist you.


Giving the Agent Hands
Introducing the Function Definition (JSON Schema).


"This is the instruction manual we give the LLM so it knows what buttons it can push."

In [16]:
import json

# 1. Define the Python Tool
def get_weather(city: str):
    # Mock database for workshop
    mock_db = {"mumbai": "32C, Humid", "delhi": "28C, Smoggy", "bangalore": "27C, Sultry"}
    return mock_db.get(city.lower(), "Data not available")

# 2. Write the TOOL SCHEMA (This is the tricky part students learn)
tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# 3. The AGENTIC CALL
def agent_ask(user_query):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": user_query}],
        tools=tool_schema, # <--- THE MAGIC LINE
        tool_choice="auto"
    )
    return response

# Test it
response = agent_ask("What's the weather like in Bangalore?")
print(response.choices[0].message)
# Output shows: tool_calls with arguments {"city": "Delhi"}

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='1hdvsyxcc', function=Function(arguments='{"city":"Bangalore"}', name='get_weather'), type='function')])


What You Expected (Wrong Mental Model):
User: "Weather in Delhi?"

↓

LLM: magically runs Python function

↓

LLM: "28C, Smoggy"

What Actually Happens (The Agentic Pattern):
User: "Weather in Delhi?"

↓

LLM: "I need to call the 'get_weather' tool with {'city': 'Delhi'}"

↓

[LLM STOPS HERE and returns the INTENT to call a tool]

↓

Python: "Oh! The LLM wants me to run get_weather('Delhi')"

↓

Python: actually executes get_weather('Delhi') → "28C, Smoggy"

↓

Python: Sends "28C, Smoggy" BACK to LLM

↓

LLM: "The weather in Delhi is 28C and Smoggy"

Step 3

In [17]:
# 3. The AGENTIC CALL (Python Executes and Feeds Back (What's Missing)
def agent_ask(user_query):
    print(f"👤 User: {user_query}")

    # STEP 1: Ask LLM (it may request a tool)
    messages = [{"role": "user", "content": user_query}]

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        tools=tool_schema,
        tool_choice="auto"
    )

    response_message = response.choices[0].message
    messages.append(response_message)

    # STEP 2: Check if LLM wants to call a tool
    if response_message.tool_calls:
        print(f"🔧 LLM wants to call: {response_message.tool_calls[0].function.name}")

        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            # STEP 3: Execute the tool
            if function_name == "get_weather":
                city = function_args.get("city")
                result = get_weather(city)
                print(f"📊 Tool result: {result}")

                # STEP 4: Feed result back to LLM
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

        # STEP 5: Get final response from LLM
        final_response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages
        )

        final_answer = final_response.choices[0].message.content
        print(f"🤖 Agent: {final_answer}")
        return final_answer
    else:
        # No tool needed - direct answer
        print(f"🤖 Agent: {response_message.content}")
        return response_message.content

# Test it
agent_ask("What's the weather like in Mumbai?")

👤 User: What's the weather like in Mumbai?
🔧 LLM wants to call: get_weather
📊 Tool result: 32C, Humid
🤖 Agent: However, please note that I'm a large language model, I don't have real-time access to current weather conditions. The weather may have changed since my knowledge cutoff in 2023.

For the most up-to-date information, I recommend checking a reliable weather website or app, such as AccuWeather or the India Meteorological Department, for the current weather conditions in Mumbai.

However, I can provide you with a general idea of the typical weather in Mumbai. Mumbai has a tropical climate, with high temperatures and high humidity throughout the year. The summer months (March to May) are particularly hot, with temperatures often reaching over 35°C. The monsoon season, which typically lasts from June to September, brings heavy rainfall and high humidity.

If you're planning a trip to Mumbai, I recommend checking the weather forecast before you go to ensure you're prepared for the c

"However, please note that I'm a large language model, I don't have real-time access to current weather conditions. The weather may have changed since my knowledge cutoff in 2023.\n\nFor the most up-to-date information, I recommend checking a reliable weather website or app, such as AccuWeather or the India Meteorological Department, for the current weather conditions in Mumbai.\n\nHowever, I can provide you with a general idea of the typical weather in Mumbai. Mumbai has a tropical climate, with high temperatures and high humidity throughout the year. The summer months (March to May) are particularly hot, with temperatures often reaching over 35°C. The monsoon season, which typically lasts from June to September, brings heavy rainfall and high humidity.\n\nIf you're planning a trip to Mumbai, I recommend checking the weather forecast before you go to ensure you're prepared for the conditions."

Here's the secret: LLMs can't run code. They can only write JSON.
When the LLM returns tool_calls, it's like it's writing a Post-it Note to Python saying 'Please run this function for me.'

Python reads the note, runs get_weather('Delhi'), and then hands the result back to the LLM saying 'Here's what you asked for.'

Only THEN does the LLM write the final answer.

This is the ENTIRE foundation of Agentic AI. The LLM is the Architect, Python is the Construction Worker."

Session 3: The ReAct Loop (Reason + Act) - The Heart of Agency

Goal: Move from single tool call to multi-step reasoning.

In [21]:
import os
from google.colab import userdata # Import userdata for Colab secrets
from groq import Groq

# Retrieve API key from Colab secrets, assuming it's named 'GROQ_API_KEY'
api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

def ask_llm(question):
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": question}]
        )
        return response.choices[0].message.content
    except Exception as e:
        # This prints the FULL error details
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Message: {str(e)}")

        # If it's an API error, get more details
        if hasattr(e, 'response'):
            print(f"Status Code: {e.response.status_code}")
            print(f"Response Body: {e.response.text}")
        return None

In [22]:
# Tool Definitions
import json

# 1. Define the Python Tool
def get_weather(city: str):
    # Mock database for workshop
    mock_db = {"mumbai": "32C, Humid", "delhi": "28C, Smoggy", "bangalore": "27C, Sultry"}
    return mock_db.get(city.lower(), "Data not available")

# 2. Write the TOOL SCHEMA (This is the tricky part students learn)
tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# 3. The AGENTIC CALL
def agent_ask(user_query):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": user_query}],
        tools=tool_schema, # <--- THE MAGIC LINE
        tool_choice="auto"
    )
    return response

In [23]:
def run_agent(user_query):
    messages = [{"role": "user", "content": user_query}]

    # The Loop - Students fill in the condition
    while True:  # <--- STUDENTS FILL THIS
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=tool_schema,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        messages.append(response_message) # Append assistant thought

        # Check if LLM wants to call a tool
        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                # EXECUTE THE PYTHON FUNCTION (Student fills this)
                if function_name == "get_weather":
                    result = get_weather(function_args.get("city"))
                    # Feed result back to LLM
                    messages.append({
                        "role": "tool",
                        "content": result,
                        "tool_call_id": tool_call.id
                    })
        else:
            # NO MORE TOOLS -> ANSWER IS READY
            return response_message.content

In [28]:
print(run_agent("Is it warmer in Mumbai or Delhi right now?"))

It is warmer in Mumbai right now.


Based on the weather information, it seems that Mumbai is warmer than Delhi right now. The temperature in Mumbai is 32C while in Delhi it is 28C.

Session 4: RAG Demo

The RAG Demo

Context: A syllabus PDF.

In [30]:
import os
from google.colab import userdata # Import userdata for Colab secrets
from groq import Groq

# Retrieve API key from Colab secrets, assuming it's named 'GROQ_API_KEY'
api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

def ask_llm(question):
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": question}]
        )
        return response.choices[0].message.content
    except Exception as e:
        # This prints the FULL error details
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Message: {str(e)}")

        # If it's an API error, get more details
        if hasattr(e, 'response'):
            print(f"Status Code: {e.response.status_code}")
            print(f"Response Body: {e.response.text}")
        return None

In [31]:
context = """
CS 202: Machine Learning
Midterm Exam: March 15th, 2026. Weight: 30%
"""

query = "When is the ML midterm?"
prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"
print(ask_llm(prompt))

March 15th, 2026.


"This is RAG. We didn't train the model on your syllabus. We just handed it the notes at test time. That's the power you get for free."